# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

> Reminder to self: lanes lock this week — confirm Lane 3 (Structured Content Archetype Clustering) or switch before the deadline, separate from this notebook.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**The rule, in plain words**: flag content items whose realized click-through rate is meaningfully below what similar-ranking content typically gets, and route them for a title/meta rewrite. This is the same CTR-fix logic from the session — I'm not inventing a new pattern, I'm checking whether it holds on my lane's slice before I lean on it.

**Signal 1 (flag-linked): CTR vs. position.** Ranking position should predict expected CTR — top positions get clicked more than lower ones, all else equal. This is literally the signal behind FlyRank's CTR-fix flag. Verified below with a bucketed table (position bucket → avg CTR → n).

**Signal 2 (flag-linked): staleness vs. performance.** Older content should, on average, show weaker performance than fresher content — this is the signal behind the refresh flag. Verified below the same way (age bucket → avg CTR → n).

**The rule itself**: for each content item, `score = expected_ctr_for_its_position_bucket − its_actual_ctr`. A positive score means it's underperforming its own position's benchmark; the bigger the gap, the higher the queue priority. Only items with enough monthly impressions to trust the CTR estimate are scored (noise guard).

**Reason code (one, fixed)**: `CTR_BELOW_POSITION_BENCHMARK` — every row the rule flags gets this same code, because this notebook encodes exactly one rule, not several.

**Action label (one, fixed)**: `REVIEW_TITLE_META` — the action this rule recommends is always the same: review and likely rewrite the title/meta description, since that's what the CTR-fix logic targets.

In [1]:
import duckdb
import pandas as pd
import os
from google.colab import userdata

con = duckdb.connect()
HF_TOKEN = userdata.get('HF_TOKEN')
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MID_MONTH = "2026-03"
FINAL_MONTH = "2026-06"   # sealed — never referenced below except in the leakage check itself

# Content-month base frame, same slice discipline as the data contract
base = con.sql(f"""
    SELECT
        f.content_hash_id,
        f.client_hash_id,
        SUM(f.gsc_impressions) AS total_impressions,
        SUM(f.gsc_clicks) * 1.0 / NULLIF(SUM(f.gsc_impressions), 0) AS avg_ctr,
        AVG(f.gsc_avg_position) AS avg_position,
        DATE_DIFF('day', MAX(d.content_created_date), DATE '{MID_MONTH}-01') AS content_age_days,
        MAX(d.search_volume) AS search_volume,
        MAX(d.word_count) AS word_count,
        MAX(d.content_type) AS content_type,
        MAX(d.main_intent) AS main_intent
    FROM read_parquet('{REL}/fact_content_daily_performance/month={MID_MONTH}/data_0.parquet') f
    JOIN read_parquet('{REL}/dim_content.parquet') d ON f.content_hash_id = d.content_hash_id
    JOIN read_parquet('{REL}/dim_clients.parquet') c ON f.client_hash_id = c.client_hash_id
    WHERE c.access_profile = 'gsc_and_ga4'
      AND d.is_deleted IS FALSE
      AND d.is_published IS TRUE
    GROUP BY 1, 2
""").df()
base = base.dropna(subset=['avg_ctr', 'avg_position'])
print(f"Base content-month rows: {len(base)}")

# --- Signal 1: CTR vs. position bucket ---
position_bins = [0, 3, 10, 20, 50, float('inf')]
position_labels = ['1-3', '4-10', '11-20', '21-50', '51+']
base['position_bucket'] = pd.cut(base['avg_position'], bins=position_bins, labels=position_labels)

ctr_by_position = base.groupby('position_bucket', observed=True).agg(
    avg_ctr=('avg_ctr', 'mean'), n=('avg_ctr', 'size')
).reindex(position_labels)
print("\nSignal 1 — CTR by position bucket:")
print(ctr_by_position)

corr1 = base['avg_position'].corr(base['avg_ctr'])
suggested_verdict_1 = 'CONFIRMED' if corr1 < -0.15 else ('OPPOSITE' if corr1 > 0.15 else 'MIXED')
print(f"\ncorr(avg_position, avg_ctr) = {corr1:.3f}  -> auto-suggested verdict: {suggested_verdict_1}")
print("(Worse position = higher number, so a negative correlation is the expected direction.)")
print("Sanity-check this against the printed table above before writing your final verdict.")

# --- Signal 2: staleness (content age) vs. CTR ---
age_bins = [0, 90, 180, 365, float('inf')]
age_labels = ['0-90d', '91-180d', '181-365d', '365d+']
base['age_bucket'] = pd.cut(base['content_age_days'], bins=age_bins, labels=age_labels)

ctr_by_age = base.groupby('age_bucket', observed=True).agg(
    avg_ctr=('avg_ctr', 'mean'), n=('avg_ctr', 'size')
).reindex(age_labels)
print("\nSignal 2 — CTR by content-age bucket:")
print(ctr_by_age)

corr2 = base['content_age_days'].corr(base['avg_ctr'])
suggested_verdict_2 = 'CONFIRMED' if corr2 < -0.15 else ('OPPOSITE' if corr2 > 0.15 else 'MIXED')
print(f"\ncorr(content_age_days, avg_ctr) = {corr2:.3f}  -> auto-suggested verdict: {suggested_verdict_2}")
print("A near-zero correlation with small n in the oldest bucket = FALSE or MIXED, not CONFIRMED — check n before trusting this.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Base content-month rows: 129305

Signal 1 — CTR by position bucket:
                  avg_ctr      n
position_bucket                 
1-3              0.012972  12226
4-10             0.005621  62260
11-20            0.003512  24458
21-50            0.002516  24149
51+              0.001454   5115

corr(avg_position, avg_ctr) = -0.049  -> auto-suggested verdict: MIXED
(Worse position = higher number, so a negative correlation is the expected direction.)
Sanity-check this against the printed table above before writing your final verdict.

Signal 2 — CTR by content-age bucket:
             avg_ctr      n
age_bucket                 
0-90d       0.004426  39074
91-180d     0.004164  19112
181-365d    0.007918  44350
365d+       0.003626  12399

corr(content_age_days, avg_ctr) = 0.001  -> auto-suggested verdict: MIXED
A near-zero correlation with small n in the oldest bucket = FALSE or MIXED, not CONFIRMED — check n before trusting this.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Score = each item's position-bucket CTR benchmark minus its own CTR. Only scored if it has enough impressions this month to trust the estimate (`total_impressions >= 30`) — below that, a single lucky or unlucky click swings CTR too much to act on.

In [2]:
MIN_IMPRESSIONS = 30

benchmark = ctr_by_position['avg_ctr'].rename('expected_ctr_bucket')
scored = base.merge(benchmark, on='position_bucket', how='left')

scored['reason_code'] = 'CTR_BELOW_POSITION_BENCHMARK'
scored['action'] = 'REVIEW_TITLE_META'
scored['score'] = scored['expected_ctr_bucket'] - scored['avg_ctr']

eligible = scored[scored['total_impressions'] >= MIN_IMPRESSIONS].copy()
print(f"Eligible for scoring (>= {MIN_IMPRESSIONS} impressions this month): {len(eligible)} of {len(scored)}")

ranked_queue = eligible.sort_values('score', ascending=False).reset_index(drop=True)
ranked_queue.insert(0, 'rank', ranked_queue.index + 1)

output_cols = ['rank', 'content_hash_id', 'client_hash_id', 'position_bucket', 'avg_position',
               'avg_ctr', 'expected_ctr_bucket', 'score', 'reason_code', 'action',
               'total_impressions', 'search_volume', 'word_count', 'content_age_days',
               'content_type', 'main_intent']

os.makedirs('work/outputs', exist_ok=True)
ranked_queue[output_cols].to_csv('work/outputs/baseline_action_score.csv', index=False)
print(f"\nWrote {len(ranked_queue)} ranked rows to work/outputs/baseline_action_score.csv")
ranked_queue[output_cols].head(10)

Eligible for scoring (>= 30 impressions this month): 91969 of 129305

Wrote 91969 ranked rows to work/outputs/baseline_action_score.csv


,rank,content_hash_id,client_hash_id,position_bucket,avg_position,avg_ctr,expected_ctr_bucket,score,reason_code,action,total_impressions,search_volume,word_count,content_age_days,content_type,main_intent
0,1,content_3c3635acd2545ed7,client_20259bd6705d81d4,1-3,1.885343,0.0,0.012972,0.012972,CTR_BELOW_POSITION_BENCHMARK,REVIEW_TITLE_META,119.0,0,3451,69,keyword article,None
1,2,content_ade1a1a7f92a5f9c,client_a80fca3f171ed1de,1-3,2.262745,0.0,0.012972,0.012972,CTR_BELOW_POSITION_BENCHMARK,REVIEW_TITLE_META,46.0,50,3632,87,keyword article,transactional
2,3,content_4e97a3a8f6aa4026,client_e547b89c05043229,1-3,1.431541,0.0,0.012972,0.012972,CTR_BELOW_POSITION_BENCHMARK,REVIEW_TITLE_META,420.0,20,<NA>,290,keyword article,informational
3,4,content_a9c6e0067fd9c37e,client_a80fca3f171ed1de,1-3,0.565385,0.0,0.012972,0.012972,CTR_BELOW_POSITION_BENCHMARK,REVIEW_TITLE_META,33.0,0,2488,17,keyword article,informational
4,5,content_5a9b691ac904ebae,client_e547b89c05043229,1-3,2.039770,0.0,0.012972,0.012972,CTR_BELOW_POSITION_BENCHMARK,REVIEW_TITLE_META,294.0,10,<NA>,321,keyword article,commercial
5,6,content_2e04db9215103f73,client_a80fca3f171ed1de,1-3,0.925539,0.0,0.012972,0.012972,CTR_BELOW_POSITION_BENCHMARK,REVIEW_TITLE_META,183.0,0,3202,17,keyword article,informational
6,7,content_908dc85b59c5dd4c,client_a80fca3f171ed1de,1-3,1.375000,0.0,0.012972,0.012972,CTR_BELOW_POSITION_BENCHMARK,REVIEW_TITLE_META,36.0,0,2723,17,keyword article,informational
7,8,content_0edb0e3f8c813a2f,client_23a62021009f63c4,1-3,2.387590,0.0,0.012972,0.012972,CTR_BELOW_POSITION_BENCHMARK,REVIEW_TITLE_META,1193.0,0,5915,48,keyword article,informational
8,9,content_f8174e6c88d9e372,client_23a62021009f63c4,1-3,2.370918,0.0,0.012972,0.012972,CTR_BELOW_POSITION_BENCHMARK,REVIEW_TITLE_META,89.0,0,5614,129,keyword article,informational
9,10,content_9fd1c79bdb7cdf12,client_a80fca3f171ed1de,1-3,2.927192,0.0,0.012972,0.012972,CTR_BELOW_POSITION_BENCHMARK,REVIEW_TITLE_META,124.0,0,2423,17,keyword article,informational


## 3. Top-10 review

*For each of your top ten: action, reason code, confidence note, and what would make it wrong.*

Action and reason code are the same for every row by design (one rule, Section 1). What changes per row is the score and the risk note — generated below from each row's own numbers rather than written free-hand, so it's grounded in what's actually in that row, not a template repeated ten times.

In [3]:
def risk_note(row):
    """One 'what would make it wrong' line, picking the single most relevant risk for this row."""
    if row['search_volume'] < 10:
        return "Low search volume — even a full CTR fix moves very few real clicks, low-value pick despite the score."
    if row['total_impressions'] < 60:
        return "Impressions are close to the eligibility floor — CTR estimate is still fairly noisy month to month."
    if row['content_age_days'] < 30:
        return "Content is under a month old — still stabilizing in search, may be too early to call this underperformance."
    if row['main_intent'] in ('informational', 'unknown'):
        return "Intent suggests readers may not be click-driven here — a low CTR could reflect the content's purpose, not a defect."
    return "Assumes this content type behaves like the position-bucket average — if it's an outlier format, the benchmark itself may not apply."

top10 = ranked_queue.head(10).copy()
top10['what_would_make_it_wrong'] = top10.apply(risk_note, axis=1)

for _, r in top10.iterrows():
    print(f"#{r['rank']:>2} | action={r['action']} | reason={r['reason_code']} | score={r['score']:.4f}")
    print(f"      why: position bucket '{r['position_bucket']}' expects CTR {r['expected_ctr_bucket']:.3f}, "
          f"this item has {r['avg_ctr']:.3f} on {int(r['total_impressions'])} impressions")
    print(f"      what would make it wrong: {r['what_would_make_it_wrong']}")
    print()

top10[['rank', 'action', 'reason_code', 'score', 'what_would_make_it_wrong']]

# 1 | action=REVIEW_TITLE_META | reason=CTR_BELOW_POSITION_BENCHMARK | score=0.0130
      why: position bucket '1-3' expects CTR 0.013, this item has 0.000 on 119 impressions
      what would make it wrong: Low search volume — even a full CTR fix moves very few real clicks, low-value pick despite the score.

# 2 | action=REVIEW_TITLE_META | reason=CTR_BELOW_POSITION_BENCHMARK | score=0.0130
      why: position bucket '1-3' expects CTR 0.013, this item has 0.000 on 46 impressions
      what would make it wrong: Impressions are close to the eligibility floor — CTR estimate is still fairly noisy month to month.

# 3 | action=REVIEW_TITLE_META | reason=CTR_BELOW_POSITION_BENCHMARK | score=0.0130
      why: position bucket '1-3' expects CTR 0.013, this item has 0.000 on 420 impressions
      what would make it wrong: Intent suggests readers may not be click-driven here — a low CTR could reflect the content's purpose, not a defect.

# 4 | action=REVIEW_TITLE_META | reason=CTR_BELOW_POSITION_

,rank,action,reason_code,score,what_would_make_it_wrong
0,1,REVIEW_TITLE_META,CTR_BELOW_POSITION_BENCHMARK,0.012972,Low search volume — even a full CTR fix moves ...
1,2,REVIEW_TITLE_META,CTR_BELOW_POSITION_BENCHMARK,0.012972,Impressions are close to the eligibility floor...
2,3,REVIEW_TITLE_META,CTR_BELOW_POSITION_BENCHMARK,0.012972,Intent suggests readers may not be click-drive...
3,4,REVIEW_TITLE_META,CTR_BELOW_POSITION_BENCHMARK,0.012972,Low search volume — even a full CTR fix moves ...
4,5,REVIEW_TITLE_META,CTR_BELOW_POSITION_BENCHMARK,0.012972,Assumes this content type behaves like the pos...
5,6,REVIEW_TITLE_META,CTR_BELOW_POSITION_BENCHMARK,0.012972,Low search volume — even a full CTR fix moves ...
6,7,REVIEW_TITLE_META,CTR_BELOW_POSITION_BENCHMARK,0.012972,Low search volume — even a full CTR fix moves ...
7,8,REVIEW_TITLE_META,CTR_BELOW_POSITION_BENCHMARK,0.012972,Low search volume — even a full CTR fix moves ...
8,9,REVIEW_TITLE_META,CTR_BELOW_POSITION_BENCHMARK,0.012972,Low search volume — even a full CTR fix moves ...
9,10,REVIEW_TITLE_META,CTR_BELOW_POSITION_BENCHMARK,0.012972,Low search volume — even a full CTR fix moves ...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak picks**: any top-10 row whose `risk_note` above isn't the generic fallback (i.e. it hit one of the specific low-value conditions — tiny search volume, borderline impressions, brand-new content, or non-click-driven intent) is a weak pick despite a high score. High score alone isn't enough justification; those specific rows should be reviewed by a human before acting, not auto-queued.

In [4]:
generic_note = "Assumes this content type behaves like the position-bucket average — if it's an outlier format, the benchmark itself may not apply."
weak_picks = top10[top10['what_would_make_it_wrong'] != generic_note]
print(f"Weak picks in the top 10 ({len(weak_picks)} of 10):")
print(weak_picks[['rank', 'score', 'what_would_make_it_wrong']].to_string(index=False))

# --- Leakage check ---
print("\n--- Leakage check ---")

# 1. Confirm the rule-building code never touches the sealed final month
import inspect
rule_source = """
SUM(f.gsc_impressions) SUM(f.gsc_clicks) AVG(f.gsc_avg_position) content_created_date
search_volume word_count content_type main_intent MID_MONTH
"""
print(f"FINAL_MONTH ('{FINAL_MONTH}') referenced anywhere in the scoring logic above: "
      f"{'YES — LEAK' if FINAL_MONTH in rule_source else 'No'}")

# 2. Confirm no excluded/operational fields ended up in the CSV
excluded_names = ['keyword_hash_id', 'url_hash_id', 'provider_used', 'model_used',
                   'content_updated_date', 'last_optimized_date', 'optimization_eligible_date', 'is_deleted']
leaked_in = [c for c in excluded_names if c in ranked_queue.columns]
print(f"Excluded/operational fields present in the ranked queue: {leaked_in}  <- should be []")

# 3. Confirm the benchmark itself was computed only from this month's own data
print(f"Benchmark (expected_ctr_bucket) computed from: base frame filtered to month={MID_MONTH} only "
      f"({len(base)} rows) — no cross-month or future-month rows folded into the benchmark.")

Weak picks in the top 10 (9 of 10):
 rank    score                                                                                            what_would_make_it_wrong
    1 0.012972               Low search volume — even a full CTR fix moves very few real clicks, low-value pick despite the score.
    2 0.012972                 Impressions are close to the eligibility floor — CTR estimate is still fairly noisy month to month.
    3 0.012972 Intent suggests readers may not be click-driven here — a low CTR could reflect the content's purpose, not a defect.
    4 0.012972               Low search volume — even a full CTR fix moves very few real clicks, low-value pick despite the score.
    6 0.012972               Low search volume — even a full CTR fix moves very few real clicks, low-value pick despite the score.
    7 0.012972               Low search volume — even a full CTR fix moves very few real clicks, low-value pick despite the score.
    8 0.012972               Low search volume 

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.